# 3D Reconstruction Pipeline
**LoFTR → COLMAP SfM (Bundle Adjustment) → Poisson Mesh**

Run each cell in order. The pipeline stages are:
1. Install & imports
2. Configuration
3. Helper functions
4. Load images
5. Load LoFTR
6. Match image pairs
7. Build COLMAP database
8. Geometric verification
9. COLMAP SfM + Bundle Adjustment
10. Build point cloud
11. Poisson mesh
12. Visualise

## Cell 1 — Install dependencies

In [ ]:
# # Run once — restart kernel after if installing for the first time
# import sys
# !{sys.executable} -m pip install "kornia==0.7.4" opencv-python open3d pycolmap --quiet

## Cell 2 — Imports

In [2]:
import os, glob, re, sqlite3
import cv2
import numpy as np
import torch
import open3d as o3d
import kornia.feature as KF
import pycolmap
from pathlib import Path

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
print(f"Open3D   : {o3d.__version__}")
print(f"pycolmap : {pycolmap.__version__ if hasattr(pycolmap, '__version__') else 'installed'}")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
PyTorch  : 2.5.1+cu121
CUDA     : True
Open3D   : 0.19.0
pycolmap : 4.0.2


## Cell 3 — Configuration
**Edit these values before running the rest of the notebook.**

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
IMAGE_DIR     = "cecum_t1_a/color"      # folder containing your input images
WORKSPACE_DIR = "colmap_workspace"   # COLMAP working directory (created automatically)
OUTPUT_PLY    = "reconstruction.ply" # output point cloud filename

# ── LoFTR ──────────────────────────────────────────────────────────────────
LOFTR_WEIGHTS  = "indoor"   # "indoor" or "outdoor"
LOFTR_MAX_SIDE = 840        # resize longest side to this before LoFTR (div-8 aligned)
MIN_CONFIDENCE = 0.3        # LoFTR match confidence threshold
LOFTR_CKPT     = 'loftr_indoor.ckpt'       # local .ckpt path to bypass SSL download, or None
                             # e.g. r"C:\Users\ariqi\loftr_indoor.ckpt"

# ── Matching ───────────────────────────────────────────────────────────────
MATCH_OVERLAP  = 3          # each frame matched against this many previous frames
FRAME_SKIP     = 2          # use every Nth frame (1=all, 3=every 3rd, 5=every 5th)

# ── Camera ─────────────────────────────────────────────────────────────────
# SIMPLE_RADIAL = COLMAP estimates focal length (good for unknown cameras)
# PINHOLE       = use your own calibrated intrinsics (set FX/FY/CX/CY below)
CAMERA_MODEL   = "SIMPLE_RADIAL"
FX, FY         = None, None  # focal lengths in pixels (None = heuristic)
CX, CY         = None, None  # principal point (None = image centre)

# TUM freiburg2 calibrated values (uncomment if using TUM):
# CAMERA_MODEL = "PINHOLE"
# FX, FY = 520.9, 521.0
# CX, CY = 325.1, 249.7

# ── Mesh ───────────────────────────────────────────────────────────────────
POISSON_DEPTH  = 9           # Poisson octree depth (higher = finer mesh, slower)

print("Configuration set.")
print(f"  IMAGE_DIR    : {IMAGE_DIR}")
print(f"  FRAME_SKIP   : {FRAME_SKIP}")
print(f"  MATCH_OVERLAP: {MATCH_OVERLAP}")
print(f"  CAMERA_MODEL : {CAMERA_MODEL}")

Configuration set.
  IMAGE_DIR    : cecum_t1_a/color
  FRAME_SKIP   : 2
  MATCH_OVERLAP: 3
  CAMERA_MODEL : SIMPLE_RADIAL


## Cell 4 — Helper functions

In [8]:
def sorted_image_paths(directory):
    """Naturally-sorted, deduplicated image paths (handles Windows case-insensitive glob)."""
    exts = ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG")
    seen, paths = set(), []
    for ext in exts:
        for p in glob.glob(os.path.join(directory, ext)):
            key = os.path.normcase(os.path.abspath(p))
            if key not in seen:
                seen.add(key)
                paths.append(p)
    paths.sort(key=lambda p: [
        int(c) if c.isdigit() else c.lower()
        for c in re.split(r"(\d+)", os.path.basename(p))
    ])
    return paths


def compute_resize(w, h, max_side):
    """Resize dims fitting max_side, snapped to nearest multiple of 8."""
    scale = max_side / max(w, h)
    nw = (int(w * scale) // 8) * 8
    nh = (int(h * scale) // 8) * 8
    return nw, nh, w / nw, h / nh


# ── COLMAP database helpers ────────────────────────────────────────────────

CAMERA_MODEL_IDS = {
    "SIMPLE_PINHOLE": 0, "PINHOLE": 1, "SIMPLE_RADIAL": 2, "RADIAL": 3,
}

def init_db(db_path):
    if os.path.exists(db_path):
        os.remove(db_path)
    conn = sqlite3.connect(db_path)
    conn.executescript("""
        CREATE TABLE cameras (
            camera_id INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
            model INTEGER NOT NULL, width INTEGER NOT NULL, height INTEGER NOT NULL,
            params BLOB, prior_focal_length INTEGER NOT NULL);
        CREATE TABLE images (
            image_id INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
            name TEXT NOT NULL UNIQUE, camera_id INTEGER NOT NULL,
            FOREIGN KEY(camera_id) REFERENCES cameras(camera_id));
        CREATE TABLE keypoints (
            image_id INTEGER PRIMARY KEY NOT NULL, rows INTEGER NOT NULL,
            cols INTEGER NOT NULL, data BLOB,
            FOREIGN KEY(image_id) REFERENCES images(image_id));
        CREATE TABLE descriptors (
            image_id INTEGER PRIMARY KEY NOT NULL, rows INTEGER NOT NULL,
            cols INTEGER NOT NULL, data BLOB,
            FOREIGN KEY(image_id) REFERENCES images(image_id));
        CREATE TABLE matches (
            pair_id INTEGER PRIMARY KEY NOT NULL, rows INTEGER NOT NULL,
            cols INTEGER NOT NULL, data BLOB);
        CREATE TABLE two_view_geometries (
            pair_id INTEGER PRIMARY KEY NOT NULL, rows INTEGER NOT NULL,
            cols INTEGER NOT NULL, data BLOB, config INTEGER NOT NULL,
            F BLOB, E BLOB, H BLOB, qvec BLOB, tvec BLOB);
    """)
    conn.commit()
    return conn


def _pair_id(id1, id2):
    if id1 > id2: id1, id2 = id2, id1
    return id1 * 2147483647 + id2


def db_add_camera(conn, model, w, h, params, prior=0):
    c = conn.execute(
        "INSERT INTO cameras(model,width,height,params,prior_focal_length) VALUES(?,?,?,?,?)",
        (model, w, h, params.astype(np.float64).tobytes(), prior))
    conn.commit()
    return c.lastrowid


def db_add_image(conn, name, cam_id):
    c = conn.execute("INSERT INTO images(name,camera_id) VALUES(?,?)", (name, cam_id))
    conn.commit()
    return c.lastrowid


def db_add_keypoints(conn, image_id, kpts):
    full = np.zeros((len(kpts), 4), dtype=np.float32)
    full[:, :2] = kpts
    full[:, 2] = 1.0
    conn.execute(
        "INSERT INTO keypoints(image_id,rows,cols,data) VALUES(?,?,?,?)",
        (image_id, len(kpts), 4, full.tobytes()))
    conn.execute(
        "INSERT INTO descriptors(image_id,rows,cols,data) VALUES(?,?,?,?)",
        (image_id, 0, 128, b""))
    conn.commit()


def db_add_matches(conn, id_a, id_b, matches):
    pid = _pair_id(id_a, id_b)
    if id_a > id_b:
        matches = matches[:, ::-1]
    conn.execute(
        "INSERT OR REPLACE INTO matches(pair_id,rows,cols,data) VALUES(?,?,?,?)",
        (pid, len(matches), 2, matches.astype(np.uint32).tobytes()))
    conn.commit()


print("Helper functions defined.")

Helper functions defined.


## Cell 5 — Load images

In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Setup workspace
ws = Path(WORKSPACE_DIR)
ws.mkdir(exist_ok=True)
(ws / "sparse").mkdir(exist_ok=True)
db_path    = str(ws / "database.db")
pairs_path = str(ws / "pairs.txt")

# Load and subsample image paths
all_paths = sorted_image_paths(IMAGE_DIR)
paths = all_paths[::FRAME_SKIP]
N = len(paths)

if N < 2:
    raise ValueError(f"Need >= 2 images in '{IMAGE_DIR}'; found {N}.")

sample = cv2.imread(paths[0])
H, W = sample.shape[:2]

print(f"Total images found : {len(all_paths)}")
print(f"After frame skip   : {N} images (every {FRAME_SKIP})")
print(f"Image size         : {W} x {H}")
print(f"First: {Path(paths[0]).name}")
print(f"Last : {Path(paths[-1]).name}")

Device: cuda
Total images found : 276
After frame skip   : 138 images (every 2)
Image size         : 1350 x 1080
First: 0_color.png
Last : 274_color.png


## Cell 6 — Load LoFTR

In [10]:
def build_loftr(device):
    if LOFTR_CKPT is not None:
        model = KF.LoFTR(pretrained=None).to(device)
        state = torch.load(LOFTR_CKPT, map_location=device, weights_only=False)
        sd = state.get("state_dict", state)
        sd = {k.replace("matcher.", ""): v for k, v in sd.items()}
        model.load_state_dict(sd, strict=False)
        return model.eval()
    return KF.LoFTR(pretrained=LOFTR_WEIGHTS).to(device).eval()


def match_pair(matcher, path_a, path_b, device):
    """Run LoFTR on two images. Returns (kpts_a, kpts_b) in full-res pixel coords."""
    def load(p):
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        h, w = img.shape[:2]
        nw, nh, sx, sy = compute_resize(w, h, LOFTR_MAX_SIDE)
        small = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
        return torch.tensor(small / 255.0, dtype=torch.float32)[None, None].to(device), (sx, sy)

    ta, sa = load(path_a)
    tb, sb = load(path_b)
    with torch.inference_mode():
        out = matcher({"image0": ta, "image1": tb})

    ka = out["keypoints0"].cpu().numpy() * np.array(sa)
    kb = out["keypoints1"].cpu().numpy() * np.array(sb)
    mask = out["confidence"].cpu().numpy() >= MIN_CONFIDENCE
    return ka[mask].astype(np.float32), kb[mask].astype(np.float32)


print(f"Loading LoFTR ({LOFTR_WEIGHTS})...")
matcher = build_loftr(device)
print(f"LoFTR loaded on: {next(matcher.parameters()).device}")

Loading LoFTR (indoor)...
LoFTR loaded on: cuda:0


## Cell 7 — Match image pairs
This is the slowest step. Each frame is matched against the previous `MATCH_OVERLAP` frames using LoFTR.

In [18]:
print(f"Matching {N} images with overlap={MATCH_OVERLAP}...")

# Storage for keypoints per image: (x,y) -> local index
image_kpts  = {p: {} for p in paths}   
matched_pairs = []  # (path_a, path_b, idx_a, idx_b)

def get_or_add(kpt_dict, xy):
    key = (round(float(xy[0]), 2), round(float(xy[1]), 2))
    if key not in kpt_dict:
        kpt_dict[key] = len(kpt_dict)
    return kpt_dict[key]

for i in range(1, N):
    for j in range(max(0, i - MATCH_OVERLAP), i):
        ka, kb = match_pair(matcher, paths[j], paths[i], device)

        print(f"  [{j:4d}]<->[{i:4d}]  "
              f"{Path(paths[j]).name} <-> {Path(paths[i]).name}  "
              f"{len(ka)} matches")

        # Skip weak pairs
        if len(ka) < 8:
            continue

        # Build consistent keypoint indices
        idx_a = np.array(
            [get_or_add(image_kpts[paths[j]], k) for k in ka],
            dtype=np.uint32
        )
        idx_b = np.array(
            [get_or_add(image_kpts[paths[i]], k) for k in kb],
            dtype=np.uint32
        )

        matched_pairs.append((paths[j], paths[i], idx_a, idx_b))

print(f"Matching complete: {len(matched_pairs)} pairs.")

Matching 138 images with overlap=3...
  [   0]<->[   1]  0_color.png <-> 2_color.png  5577 matches
  [   0]<->[   2]  0_color.png <-> 4_color.png  5167 matches
  [   1]<->[   2]  2_color.png <-> 4_color.png  5446 matches
  [   0]<->[   3]  0_color.png <-> 6_color.png  4716 matches
  [   1]<->[   3]  2_color.png <-> 6_color.png  4982 matches
  [   2]<->[   3]  4_color.png <-> 6_color.png  5457 matches
  [   1]<->[   4]  2_color.png <-> 8_color.png  4566 matches
  [   2]<->[   4]  4_color.png <-> 8_color.png  5110 matches
  [   3]<->[   4]  6_color.png <-> 8_color.png  5546 matches
  [   2]<->[   5]  4_color.png <-> 10_color.png  4919 matches
  [   3]<->[   5]  6_color.png <-> 10_color.png  5342 matches
  [   4]<->[   5]  8_color.png <-> 10_color.png  5509 matches
  [   3]<->[   6]  6_color.png <-> 12_color.png  5006 matches
  [   4]<->[   6]  8_color.png <-> 12_color.png  5192 matches
  [   5]<->[   6]  10_color.png <-> 12_color.png  5548 matches
  [   4]<->[   7]  8_color.png <-> 14_co

## Cell 8 — Build COLMAP database

In [28]:
# Fix: COLMAP marks many pairs as config=6 (panoramic/degenerate rotation)
# and skips them during SfM initialisation. We override all configs to 2
# (fundamental matrix) so COLMAP treats every verified pair as usable.
# The pairs already have correct inlier matches — we're just changing the label.

conn_fix = sqlite3.connect(db_path)

# Count configs before fix
configs_before = conn_fix.execute(
    'SELECT config, COUNT(*) FROM two_view_geometries GROUP BY config'
).fetchall()
print('Configs before fix:')
config_names = {1:'degenerate', 2:'fundamental', 3:'essential', 4:'homography', 6:'panoramic'}
for cfg, cnt in configs_before:
    print(f'  config {cfg} ({config_names.get(cfg, "unknown")}): {cnt} pairs')

# Override all panoramic (6) to fundamental (2)
n_fixed = conn_fix.execute(
    'UPDATE two_view_geometries SET config=2 WHERE config=6'
).rowcount
conn_fix.commit()
conn_fix.close()

print(f'\Fixed {n_fixed} panoramic pairs -> fundamental.')
print('Database ready for SfM.')

Configs before fix:
\Fixed 0 panoramic pairs -> fundamental.
Database ready for SfM.


In [29]:
# ── DEBUG: Check matches + database ───────────────────────────────
print("\n[DEBUG] Checking match counts...")

print("Total matched_pairs:", len(matched_pairs))

conn_dbg = sqlite3.connect(db_path)

num_matches = conn_dbg.execute(
    "SELECT COUNT(*) FROM matches"
).fetchone()[0]

print("DB matches table:", num_matches)

conn_dbg.close()


[DEBUG] Checking match counts...
Total matched_pairs: 408
DB matches table: 408


## Cell 9 — Geometric verification
Runs epipolar RANSAC on each pair and writes results to `two_view_geometries` — the table COLMAP's mapper actually reads.

In [ ]:
print("Running geometric verification...")
pycolmap.verify_matches(db_path, pairs_path)
print("Geometric verification done.")

Running geometric verification...
Geometric verification done.


In [33]:
conn_dbg = sqlite3.connect(db_path)

rows = conn_dbg.execute(
    "SELECT rows FROM two_view_geometries"
).fetchall()

print("\n[DEBUG] Inliers per pair (first 20):")
print([r[0] for r in rows[:20]])

print("Total verified pairs:", len(rows))

conn_dbg.close() 


[DEBUG] Inliers per pair (first 20):
[]
Total verified pairs: 0


## Cell 9b — Fix degenerate pair configs
COLMAP skips `config=6` (panoramic) pairs during initialisation. This cell overrides them to `config=2` so all verified pairs are used.

In [31]:
conn_fix = sqlite3.connect(db_path)

configs_before = conn_fix.execute(
    'SELECT config, COUNT(*) FROM two_view_geometries GROUP BY config'
).fetchall()

print('Configs before fix:')
config_names = {1:'degenerate', 2:'fundamental', 3:'essential', 4:'homography', 6:'panoramic'}
for cfg, cnt in configs_before:
    print(f'  config {cfg} ({config_names.get(cfg, "unknown")}): {cnt} pairs')

n_fixed = conn_fix.execute(
    'UPDATE two_view_geometries SET config=2 WHERE config=6'
).rowcount

conn_fix.commit()
conn_fix.close()

print(f'Fixed {n_fixed} panoramic pairs -> fundamental.')

Configs before fix:
Fixed 0 panoramic pairs -> fundamental.


## Cell 10 — COLMAP SfM + Bundle Adjustment

In [32]:
print("Running COLMAP incremental SfM + bundle adjustment...")

sfm_opts = pycolmap.IncrementalPipelineOptions()
sfm_opts.min_num_matches                 = 6    # min matches to use a pair
sfm_opts.min_model_size                  = 3    # min cameras to keep a model
sfm_opts.mapper.init_min_num_inliers     = 8   # easier initialisation
sfm_opts.mapper.abs_pose_min_num_inliers = 8   # easier frame registration
sfm_opts.mapper.init_max_error           = 6.0  # looser init threshold
sfm_opts.mapper.abs_pose_max_error       = 16.0 # looser registration threshold
sfm_opts.mapper.filter_max_reproj_error  = 6.0  # keep more triangulated points

recons = pycolmap.incremental_mapping(
    database_path = db_path,
    image_path    = IMAGE_DIR,
    output_path   = str(ws / "sparse"),
    options       = sfm_opts,
)

if not recons:
    raise RuntimeError(
        "COLMAP SfM failed — no reconstruction produced."
        "Try: lower FRAME_SKIP, lower MIN_CONFIDENCE, or check IMAGE_DIR."
    )

best_recon = max(recons.values(), key=lambda r: len(r.images))
best_recon.write(str(ws / "sparse" / "0"))

print(f"SfM complete!")
print(f"  Reconstructed cameras : {len(best_recon.images)}")
print(f"  Sparse 3D points      : {len(best_recon.points3D):,}")
print(f"  Saved to              : {ws / 'sparse' / '0'}")

Running COLMAP incremental SfM + bundle adjustment...


RuntimeError: COLMAP SfM failed — no reconstruction produced.Try: lower FRAME_SKIP, lower MIN_CONFIDENCE, or check IMAGE_DIR.

## Cell 11 — Build point cloud

In [ ]:
pts  = np.array([p.xyz       for p in best_recon.points3D.values()])
cols = np.array([p.color / 255.0 for p in best_recon.points3D.values()])

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(pts)
pcd.colors = o3d.utility.Vector3dVector(cols)
print(f"Raw sparse cloud: {len(pcd.points):,} points")

# Statistical outlier removal
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
print(f"After outlier removal: {len(pcd.points):,} points")

o3d.io.write_point_cloud(OUTPUT_PLY, pcd)
print(f"Saved → {OUTPUT_PLY}")

## Cell 12 — Poisson mesh

In [ ]:
print("Estimating normals...")
pcd.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
pcd.orient_normals_consistent_tangent_plane(100)

print(f"Running Poisson reconstruction (depth={POISSON_DEPTH})...")
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
    pcd, depth=POISSON_DEPTH)

# Trim low-density artefact vertices
dens = np.asarray(densities)
mesh.remove_vertices_by_mask(dens < np.quantile(dens, 0.05))

mesh_path = OUTPUT_PLY.replace(".ply", "_mesh.ply")
o3d.io.write_triangle_mesh(mesh_path, mesh)

print(f"Saved mesh → {mesh_path}")
print(f"  Vertices : {len(mesh.vertices):,}")
print(f"  Triangles: {len(mesh.triangles):,}")

## Cell 13 — Visualise
Opens an interactive Open3D viewer window (requires a display).

In [ ]:
print("Opening 3D viewer... (close window to continue)")
o3d.visualization.draw_geometries(
    [mesh],
    window_name="3D Reconstruction",
    width=1280, height=720,
)